In [ ]:
import io
from pathlib import Path
from hashlib import md5
from typing import Literal

import pandas as pd
import requests
from tqdm import trange
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

GDC_BASE = "https://api.gdc.cancer.gov"
OUT_DIR = Path("../data/RNAseq")
COHORT_T = Literal["TCGA-LUAD", "TCGA-LUSC", "CPTAC-LUAD", "CPTAC-LSCC"]

In [ ]:
def make_filter(cohort: COHORT_T):
    match cohort:
        case "TCGA-LUAD":
            project = "TCGA-LUAD"
            disease_type = "adenomas and adenocarcinomas"
        case "TCGA-LUSC":
            project = "TCGA-LUSC"
            disease_type = "squamous cell neoplasms"
        case "CPTAC-LUAD":
            project = "CPTAC-3"
            disease_type = "adenomas and adenocarcinomas"
        case "CPTAC-LSCC":
            project = "CPTAC-3"
            disease_type = "squamous cell neoplasms"
        case _:
            raise ValueError(f"Unknown project/disease_type values for {cohort}")

    return {
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
            {"op": "in", "content": {"field": "cases.disease_type", "value": [disease_type]}},
            {"op": "in", "content": {"field": "cases.primary_site", "value": ["bronchus and lung"]}},
            {"op": "in", "content": {"field": "cases.samples.tissue_type", "value": ["tumor"]}},
            {"op": "in", "content": {"field": "files.experimental_strategy", "value": ["RNA-Seq"]}},
            {"op": "in", "content": {"field": "files.data_format", "value": ["tsv"]}},
            {"op": "in", "content": {"field": "files.access", "value": ["open"]}},
            {"op": "in", "content": {"field": "cases.samples.tumor_descriptor", "value": ["primary"]}},
        ],
    }

In [ ]:
def make_session(
    total_retries=5,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
):
    session = requests.Session()
    retry = Retry(
        total=total_retries,
        read=total_retries,
        connect=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(["GET", "POST"]),
        raise_on_status=False,
        respect_retry_after_header=True,  # honor 429 Retry-After
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

In [ ]:
def download_data(cohort: COHORT_T):
    resp = requests.post(
        f"{GDC_BASE}/files",
        json={
            "filters": make_filter(cohort),
            "fields": ",".join(["file_name", "cases.submitter_id", "experimental_strategy", "data_format", "md5sum"]),
            "format": "TSV",
            "size": "10000", # the LUAD/LSCC cohorts shouldn't be larger than this
        },
    )
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text), sep="\t")

    cohort_dir = OUT_DIR / cohort
    cohort_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(cohort_dir / "metadata.csv", index=False)

    tsv_dir = cohort_dir / "TSVs"
    tsv_dir.mkdir(exist_ok=True)
    session = make_session()
    for i in trange(len(df), desc=f"Downloading {cohort}"):
        fname = df.loc[i, "file_name"]
        fid = df.loc[i, "id"]
        fhash = df.loc[i, "md5sum"]
        case_id = df.loc[i, "cases.0.submitter_id"]
        case_dir = tsv_dir / case_id # type: ignore
        case_dir.mkdir(exist_ok=True)
        fpath = case_dir / fname # type: ignore

        _hash = "NONE"
        if fpath.exists():
            with open(fpath, "rb") as f:
                _hash = md5(f.read()).hexdigest()

        # no need to redownload if hash matches
        if fhash != _hash:
            temp = session.get(f"{GDC_BASE}/data/{fid}")
            temp.raise_for_status()
            with open(fpath, "w") as f:
                f.write(temp.text)
            _hash = md5(temp.content).hexdigest()
            assert fhash == _hash

In [ ]:
download_data("TCGA-LUAD")
download_data("TCGA-LUSC")
download_data("CPTAC-LUAD")
download_data("CPTAC-LSCC")